# ODS v2: Customer interactions

Builds 10 ods tables in `ods_v2`: `CallSession`, `TelephonyCall`, `MessageEvent`, `Campaign`, `ConsentAction`, `CdaMessage`, `WebPageView`, `WebQuoteClick`, `WebRetrieveHit`, `SourceCustomer`.

Needs these notebooks to have succeeded first: 20_Quote.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("30_Interaction", "started")


## CallSession


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.CallSession                                                                  */
/*  Description : Genesys sessions (all media) with routing, outcome and durations. The customer's numb */
/*              : ers (ani / dnis) are salted                                                           */
/*              : PhoneKeys; agent names and from / to addresses are not taken. ParticipantName is kept */
/*              :  only for system / queue                                                              */
/*              : participants (it can be the customer's name on customer / external legs). The *IsMobi */
/*              : le flags say whether each                                                             */
/*              : number is an Irish mobile (national form 08...), set before hashing (notebook 04 cell */
/*              :  165 filters on it).                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per Genesys conversation and session                                   */
/*              : Business key: ConversationId + SessionId                                              */
/*              : History: event   Cut-off: ConversationStartTime                                       */
/*              : Source: dlk.genesys_session_summary                                                   */
/*              : Reads: dlk.genesys_session_summary                                                    */
/******************************************************************************************************/
create or replace table ${ods.target}.CallSession as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  g.conversationId                                                as ConversationId,
                g.sessionId                                                     as SessionId,
                g.sessionIndex                                                  as SessionIndex,
                g.participantid                                                 as ParticipantId,
                g.conversationStartTime                                         as ConversationStartTime,
                g.conversationEndTime                                           as ConversationEndTime,
                g.sessionStartTime                                              as SessionStartTime,
                g.sessionEndTime                                                as SessionEndTime,
                g.mediaType                                                     as MediaType,
                g.messageType                                                   as MessageType,
                g.direction                                                     as Direction,
                g.originatingDirection                                          as OriginatingDirection,
                g.purpose                                                       as Purpose,
                g.queueId                                                       as QueueId,
                g.queueName                                                     as QueueName,
                case when lower(g.purpose) in ('customer', 'external') then null else g.participantName end
                                                                                as ParticipantName,
                g.campaignName                                                  as CampaignName,
                g.userId                                                        as AgentUserId,
                g.abandoned                                                     as Abandoned,
                g.agentAnswered                                                 as AgentAnswered,
                g.alertNoAnswer                                                 as AlertNoAnswer,
                g.offered                                                       as Offered,
                g.queueAnswered                                                 as QueueAnswered,
                g.transferred                                                   as Transferred,
                g.blindtransferred                                              as BlindTransferred,
                g.consulttransferred                                            as ConsultTransferred,
                ods_redact(g.transferToId)                                          as TransferToId,       -- external transfers carry a number
                g.transferToPurpose                                             as TransferToPurpose,
                g.disconnectType                                                as DisconnectType,
                g.wrapupCode                                                    as WrapupCode,
                g.wrapupCodeName                                                as WrapupCodeName,
                g.sessionDuration                                               as SessionDurationMs,
                g.totalAcdWaitDuration                                          as AcdWaitDurationMs,
                g.totalAgentAlertDuration                                       as AgentAlertDurationMs,
                g.totalAgentTalkDuration                                        as AgentTalkDurationMs,
                g.totalAgentHoldDuration                                        as AgentHoldDurationMs,
                g.totalAgentWrapupDuration                                      as AgentWrapupDurationMs,
                g.holdcount                                                     as HoldCount,
                g.callbackCount                                                 as CallbackCount,
                ods_hash_phone(replace(g.ani, 'tel:', ''))                          as AniPhoneKey,
                ods_hash_phone(replace(g.dnis, 'tel:', ''))                         as DnisPhoneKey,
                ods_hash_phone(replace(g.originatingDnis, 'tel:', ''))              as OriginatingDnisPhoneKey,
                case when regexp_replace(regexp_replace(regexp_replace(regexp_replace(cast(replace(g.ani, 'tel:', '') as string), '[^0-9]', ''), '^00353', '0'), '^353', '0'), '^(8[0-9]{8})$', '0$1') like '08%' then 1 else 0 end                           as AniIsMobile,
                case when regexp_replace(regexp_replace(regexp_replace(regexp_replace(cast(replace(g.dnis, 'tel:', '') as string), '[^0-9]', ''), '^00353', '0'), '^353', '0'), '^(8[0-9]{8})$', '0$1') like '08%' then 1 else 0 end                          as DnisIsMobile,
                case when regexp_replace(regexp_replace(regexp_replace(regexp_replace(cast(replace(g.originatingDnis, 'tel:', '') as string), '[^0-9]', ''), '^00353', '0'), '^353', '0'), '^(8[0-9]{8})$', '0$1') like '08%' then 1 else 0 end               as OriginatingDnisIsMobile
        from    ${ods.dlk}.genesys_session_summary g
        where   g.conversationStartTime >= date '${ods.cutoff_date}'
        ) x

## TelephonyCall


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.TelephonyCall                                                                */
/*  Description : The warehouse's call detail: call legs with work group, disposition and the MFQ / HEQ */
/*              :  quote and client the                                                                 */
/*              : warehouse matched each call to. Phone numbers are salted PhoneKeys; agent names are n */
/*              : ot taken. To be derived                                                               */
/*              : from dlk.genesys_session_summary with the DWH logic later.                            */
/*              :                                                                                       */
/*              : Grain: one row per call id and sequence                                               */
/*              : Business key: CallIDKey + SeqNo                                                       */
/*              : History: event   Cut-off: CallStartDT                                                 */
/*              : Source: edw.tbl_fact_telephony_call_detail (derived in the warehouse from Genesys; se */
/*              : e DWH Build)                                                                          */
/*              : Reads: edw.tbl_fact_telephony_call_detail                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.TelephonyCall as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  t.CallIDKey,
                t.SeqNo,
                t.CallStartDT,
                t.CallConnectedDT,
                t.CallTerminatedDT,
                t.CallType,
                t.CallDirection,
                t.WorkGroup,
                t.Disposition,
                t.DispositionCode,
                t.WrapUpCodeName,
                t.MediaType,
                t.ConnectedDuration,
                t.HoldDuration,
                t.CallDuration,
                t.CampaignCall,
                t.CampaignName,
                t.CampaignWrapUpCode,
                t.AgentCode,
                t.WFPersonID                                                    as AgentPersonId,
                t.ExtTeamName,
                t.ExtDepartment,
                nullif(upper(trim(t.ClientCode)), '')                                              as ClientCode,
                t.OpportunityPolicyTypeGroup,
                t.NewBusIBFlag,
                t.NewBusOBFlag,
                t.RenewalsIBFlag,
                t.MTAIBFlag,
                nullif(lower(trim(t.MFQQuoteQueryGuid)), '')                                       as MfqGuidKey,
                t.MFQCreatedDatetime,
                t.MFQQuoteType,
                t.MFQMatchType,
                case when t.HEQQuotationID like '%@%' then ods_hash_id(t.HEQQuotationID, '')      -- some ids are email + timestamp
                     else nullif(upper(trim(t.HEQQuotationID)), '') end                            as HeqQuotationId,
                t.HEQCreatedDateTime,
                t.HEQMatchType,
                t.GenesysLeadID,
                ods_hash_phone(t.PhoneNumber)                                       as PhoneKey
        from    ${ods.edw}.tbl_fact_telephony_call_detail t
        where   t.CallStartDT >= date '${ods.cutoff_date}'
        ) x

## MessageEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_xp_events                                                            */
/*  Description : Both Xtremepush feeds in one shape, contact details hashed.                           */
/*              :                                                                                       */
/*              : Reads: dlk.ext_xtremepushresults, dlk.ext_xtremepushresults_policy                    */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_xp_events as
select  'Quote' as SourceFeed, campaign_id, campaign_name, campaign_variant, profile_id, interaction_type, interaction_value,
        `timestamp`, PolicyCode, ClientCode, QuoteQueryGuid, workflow_node, MessageType, SourceFileDate, `$ExtractDateTime` as ExtractDateTime,
        ods_hash_email(email) as EmailKey, ods_hash_phone(mobile_number) as MobileKey, ods_hash_id(user_id, '') as UserIdKey
from    ${ods.dlk}.ext_xtremepushresults
where   `timestamp` >= date '${ods.cutoff_date}'
union all
select  'Policy', campaign_id, campaign_name, campaign_variant, profile_id, interaction_type, interaction_value,
        `timestamp`, PolicyCode, ClientCode, QuoteQueryGuid, workflow_node, MessageType, SourceFileDate, `$ExtractDateTime`,
        ods_hash_email(email), ods_hash_phone(mobile_number), ods_hash_id(user_id, '')
from    ${ods.dlk}.ext_xtremepushresults_policy
where   `timestamp` >= date '${ods.cutoff_date}'

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.MessageEvent                                                                 */
/*  Description : Xtremepush message events (sent / open / click / bounce ...) from the quote and polic */
/*              : y feeds, one row per event:                                                           */
/*              : the same event loaded from several files is kept once (latest file). Email, mobile an */
/*              : d the Xtremepush user id                                                              */
/*              : are salted hash keys.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per message event (feed, campaign, profile, interaction, time, value,  */
/*              : variant, node, message type)                                                          */
/*              : Business key: SourceFeed + CampaignId + ProfileId + InteractionType + EventDateTime + */
/*              :  InteractionValue + CampaignVariant + WorkflowNode + MessageType                      */
/*              : History: event   Cut-off: EventDateTime                                               */
/*              : Source: dlk.ext_xtremepushresults + dlk.ext_xtremepushresults_policy                  */
/*              : Reads: stg.odsv2_xp_events                                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.MessageEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.SourceFeed,
                x.campaign_id                                                   as CampaignId,
                x.campaign_name                                                 as CampaignName,
                x.campaign_variant                                              as CampaignVariant,
                x.profile_id                                                    as ProfileId,
                x.interaction_type                                              as InteractionType,
                ods_redact(x.interaction_value)                                     as InteractionValue,
                x.`timestamp`                                                   as EventDateTime,
                x.workflow_node                                                 as WorkflowNode,
                x.MessageType,
                nullif(upper(trim(x.PolicyCode)), '')                                              as PolicyCode,
                nullif(upper(trim(x.ClientCode)), '')                                              as ClientCode,
                nullif(lower(trim(x.QuoteQueryGuid)), '')                                          as QuoteGuidKey,
                x.EmailKey,
                x.MobileKey,
                x.UserIdKey,
                x.SourceFileDate
        from    (
                select  *,
                        row_number() over (partition by SourceFeed, campaign_id, profile_id, interaction_type, `timestamp`,
                                                        interaction_value, campaign_variant, workflow_node, MessageType
                                           order by SourceFileDate desc, ExtractDateTime desc) as rn
                from    ${ods.stg}.odsv2_xp_events
                ) x
        where   x.rn = 1
        ) x

## Campaign


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Campaign                                                                     */
/*  Description : Campaign names versioned by date: each name a campaign id has carried, with the first */
/*              :  and last message event under                                                         */
/*              : it. NameCount > 1 means the campaign was renamed; NamesOverlap = 1 flags names used i */
/*              : n overlapping periods (then                                                           */
/*              : the dates do not separate them cleanly).                                              */
/*              :                                                                                       */
/*              : Grain: one row per campaign id and campaign name                                      */
/*              : Business key: CampaignId + CampaignName   Surrogate key: CampaignVersionId            */
/*              : History: versioned (ValidFrom / ValidTo = first / last event under that name)   Cut-o */
/*              : ff: none                                                                              */
/*              : Source: ods MessageEvent (Xtremepush campaigns)                                       */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.Campaign as
select  row_number() over (order by CampaignId, CampaignName)       as CampaignVersionId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.CampaignId,
                c.CampaignName,
                c.ValidFrom,
                c.ValidTo,
                c.EventCount,
                count(*) over (partition by c.CampaignId)                       as NameCount,
                case when lead(c.ValidFrom) over (partition by c.CampaignId order by c.ValidFrom, c.CampaignName) <= c.ValidTo
                       or lag(c.ValidTo) over (partition by c.CampaignId order by c.ValidFrom, c.CampaignName) >= c.ValidFrom
                     then 1 else 0 end                                          as NamesOverlap,
                case when row_number() over (partition by c.CampaignId order by c.ValidTo desc, c.CampaignName) = 1
                     then 1 else 0 end                                          as IsLatestName
        from    (
                select  CampaignId, CampaignName, min(EventDateTime) as ValidFrom, max(EventDateTime) as ValidTo, count(*) as EventCount
                from    ${ods.target}.MessageEvent
                where   CampaignId is not null
                group by CampaignId, CampaignName
                ) c
        ) x

## ConsentAction


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.ConsentAction                                                                */
/*  Description : Every marketing consent action recorded in Mercury (all history, not only the latest  */
/*              : per contact): channel,                                                                */
/*              : action, when, and the app that recorded it. The contact (email for channel 3, phone f */
/*              : or channels 1 / 2) is a                                                               */
/*              : salted EmailKey / PhoneKey, so ads can link consent to MFQ quotes and SCV keys by ema */
/*              : il point in time.                                                                     */
/*              :                                                                                       */
/*              : Grain: one row per Mercury consent action                                             */
/*              : Business key: ActionTrackId                                                           */
/*              : History: event (no cut-off; consent is a state, its full history is needed)   Cut-off */
/*              : : none                                                                                */
/*              : Source: dlk.merc_actiontracks + merc_servicecallstracks + merc_apps                   */
/*              : Reads: dlk.merc_actiontracks, dlk.merc_servicecallstracks, dlk.merc_apps              */
/******************************************************************************************************/
create or replace table ${ods.target}.ConsentAction as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  a.ActionTrackId,
                a.ActionId,
                a.ChannelId,
                case when a.ChannelId = 3 then 'Email' when a.ChannelId in (1, 2) then 'Phone' else 'Other' end
                                                                                as ContactType,
                a.`DateTime`                                                    as ConsentRecordedDateTime,
                a.EffectiveDateTime,
                a.ServiceCallsTrackId,
                a.BrokerId,
                s.AppId,
                p.Name                                                          as AppName,
                s.SubApp,
                s.isServiceExecuted                                             as IsServiceExecuted,
                case when a.ChannelId = 3 then ods_hash_email(a.Value) end          as EmailKey,
                case when a.ChannelId in (1, 2) then ods_hash_phone(a.Value) end    as PhoneKey
        from    ${ods.dlk}.merc_actiontracks            a
        left join ${ods.dlk}.merc_servicecallstracks    s   on s.ServiceCallsTrackId = a.ServiceCallsTrackId
        left join ${ods.dlk}.merc_apps                  p   on p.AppId = s.AppId
        ) x

## CdaMessage


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.CdaMessage                                                                   */
/*  Description : CDA (SMS / email) messages sent per client and policy: campaign, batch, status and pr */
/*              : ovider response, and the                                                              */
/*              : message type. The body is read only to classify it (7 patterns covered all 116,711 me */
/*              : ssages of Jun-Aug 2026); it is                                                        */
/*              : not held (it carries names). Every message carries its policy number or client code i */
/*              : n its own column, so no code                                                          */
/*              : is read from the text. 99 = the SMS engine's own daily test message. Contact and emai */
/*              : l are salted keys.                                                                    */
/*              :                                                                                       */
/*              : Grain: one row per CDA message                                                        */
/*              : Business key: MessageId                                                               */
/*              : History: event   Cut-off: SentDateTime                                                */
/*              : Source: dlk.sql07cda_messages (read from dlk_bak_20260914 until it lands in dlk)      */
/*              : Reads: dlk_bak_20260914.sql07cda_messages                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.CdaMessage as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  m.MessageId,
                case
                     when m.RenderedMessage like '%FQ Chase%' then 1
                     when m.RenderedMessage like '%emailed important correspondence%' then 2
                     when m.RenderedMessage like '%thanks for choosing us%' then 3
                     when m.RenderedMessage like '%thank you for choosing Chill Insurance. We have issued%' then 4
                     when m.RenderedMessage like '%renewal is due tomorrow%' then 5
                     when m.RenderedMessage like '%renewal is now due%' then 6
                     when m.RenderedMessage like '%SMS engine is OK%' then 99
                     else 0 end                                                 as MessageTypeId,
                case
                     when m.RenderedMessage like '%FQ Chase%' then 'MFQ chase'
                     when m.RenderedMessage like '%emailed important correspondence%' then 'Correspondence emailed'
                     when m.RenderedMessage like '%thanks for choosing us%' then 'Welcome'
                     when m.RenderedMessage like '%thank you for choosing Chill Insurance. We have issued%' then 'Policy issued'
                     when m.RenderedMessage like '%renewal is due tomorrow%' then 'Renewal due tomorrow'
                     when m.RenderedMessage like '%renewal is now due%' then 'Renewal now due'
                     when m.RenderedMessage like '%SMS engine is OK%' then 'SMS engine test'
                     else 'Unclassified' end                                    as MessageType,
                m.DailyCampaignId,
                m.BatchId,
                m.SentDateTime,
                m.MessageStatus,
                m.ProviderResponse,
                m.ProviderResponseDateTime,
                m.UTCDateAdded,
                m.UTCDateUpdated,
                m.Priority,
                nullif(upper(trim(m.ClientCode)), '')                                              as ClientCode,
                nullif(upper(trim(m.PolicyNumber)), '')                                            as PolicyCode,
                m.IsSendViaChillPush,
                case when m.Contact like '%@%' then ods_hash_email(m.Contact) else ods_hash_phone(m.Contact) end
                                                                                as ContactKey,
                ods_hash_email(m.Email)                                             as EmailKey
        from    ${ods.dlk_cda}.sql07cda_messages m
        where   m.SentDateTime >= date '${ods.cutoff_date}'
        ) x

## WebPageView


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_ga4_quote_sessions                                                   */
/*  Description : GA4 sessions with at least one motor or home quote reference.                         */
/*              :                                                                                       */
/*              : Reads: dlk.ga4_page_views_daily                                                       */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_ga4_quote_sessions as
select  distinct SessionID
from    ${ods.dlk}.ga4_page_views_daily
where   EventDate >= date '${ods.cutoff_date}'
and     (nullif(trim(MotorQuoteReference), '') is not null or nullif(trim(HomeQuoteReference), '') is not null)

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.WebPageView                                                                  */
/*  Description : GA4 page views in sessions that carry a motor or home quote reference (the sessions t */
/*              : he assets use: device,                                                                */
/*              : source, pages around a quote). PagePathCleaned only; the raw path with query string i */
/*              : s not taken.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per GA4 page view (client, click order, effective date, event time)    */
/*              : Business key: ClientID + ClickOrder + EffectiveDate + EventDateTime                   */
/*              : History: event   Cut-off: EventDate                                                   */
/*              : Source: dlk.ga4_page_views_daily                                                      */
/*              : Reads: dlk.ga4_page_views_daily, stg.odsv2_ga4_quote_sessions                         */
/******************************************************************************************************/
create or replace table ${ods.target}.WebPageView as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  g.ClientID,
                g.ClickOrder,
                g.EffectiveDate,
                g.EventDate,
                g.EventDateTime,
                g.EventName,
                g.SessionID,
                g.SessionDuration,
                g.SessionEndDateTime,
                g.SessionGroup,
                g.SessionCount,
                g.GASource,
                g.GAMedium,
                g.DeviceCategory,
                g.PageTitle,
                g.PagePathCleaned,
                g.UserPseudoID,
                g.CampaignID,
                g.CampaignName,
                g.Keyword,
                g.PageViewDuration,
                g.MotorQuoteID,
                nullif(upper(trim(g.MotorQuoteReference)), '')                                     as MotorQuoteReference,
                nullif(upper(trim(g.HomeQuoteReference)), '')                                      as HomeQuoteReference,
                g.OrgID
        from    ${ods.dlk}.ga4_page_views_daily     g
        join    ${ods.stg}.odsv2_ga4_quote_sessions s   on s.SessionID = g.SessionID
        where   g.EventDate >= date '${ods.cutoff_date}'
        ) x

## WebQuoteClick


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.WebQuoteClick                                                                */
/*  Description : GA4 clicks that carry an MFQ quote guid (the five MFQ-QuoteQueryGUID columns turned i */
/*              : nto rows), with source,                                                               */
/*              : medium, campaign and device, as ODS_Build_QuoteJourney GAQuoteClicks.                 */
/*              :                                                                                       */
/*              : Grain: one row per MFQ guid and GA click                                              */
/*              : Business key: GuidKey + ClientID + ClickOrder + EffectiveDate                         */
/*              : History: event   Cut-off: WorkDate                                                    */
/*              : Source: edw.tbl_fact_ga_clicks_ga4                                                    */
/*              : Reads: edw.tbl_fact_ga_clicks_ga4, ods_v2.Quote                                       */
/******************************************************************************************************/
create or replace table ${ods.target}.WebQuoteClick as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  distinct
                q.QuoteId,
                nullif(lower(trim(g.guid)), '')                                                    as GuidKey,
                g.ClientID,
                g.ClickOrder,
                g.EffectiveDate,
                g.WorkDate,
                g.WorkDateTime                                                  as WorkDateTimeUtc,
                g.GASource,
                g.GAMedium,
                g.Campaign,
                g.DeviceCategory,
                g.FirstDevice,
                ods_redact(g.PagePath)                                              as PagePath
        from    (
                select  c.*, s.guid
                from    ${ods.edw}.tbl_fact_ga_clicks_ga4 c
                lateral view stack(5, `MFQ-QuoteQueryGUID1`, `MFQ-QuoteQueryGUID2`, `MFQ-QuoteQueryGUID3`,
                                      `MFQ-QuoteQueryGUID4`, `MFQ-QuoteQueryGUID5`) s as guid
                where   c.WorkDate >= date '${ods.cutoff_date}'
                ) g
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(g.guid)), '')
        where   nullif(trim(g.guid), '') is not null
        ) x

## WebRetrieveHit


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.WebRetrieveHit                                                               */
/*  Description : Car retrieve-page hits with ?reference= and the reference they carried (paths as note */
/*              : book 05 cells 11-12). Raw                                                             */
/*              : hits only: resolving each hit to the quote it opened (first quote with that reference */
/*              :  created 5 s before to 30 min                                                         */
/*              : after the hit; references are reused after 30-60 days) is a derived relationship and  */
/*              : belongs in ads.                                                                       */
/*              :                                                                                       */
/*              : Grain: one row per retrieve-page hit with a ?reference= (client, click order, time)   */
/*              : Business key: ClientID + ClickOrder + HitDateTimeUtc                                  */
/*              : History: event   Cut-off: HitDateTimeUtc                                              */
/*              : Source: edw.tbl_fact_ga_clicks_ga4                                                    */
/*              : Reads: edw.tbl_fact_ga_clicks_ga4                                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.WebRetrieveHit as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  ClientID,
                ClickOrder,
                HitDateTimeUtc,
                ods_redact(PagePath)                                                as PagePath,
                Reference
        from    (
                select  ClientID,
                        ClickOrder,
                        WorkDateTime                                            as HitDateTimeUtc,
                        PagePath,
                        nullif(upper(trim(regexp_extract(regexp_extract(PagePath, '\\?([^#]*)', 1),
                                                         '(?i)(?:^|&)reference=([^&]*)', 1))), '')
                                                                                as Reference,
                        row_number() over (partition by ClientID, ClickOrder, WorkDateTime order by EffectiveDate desc) as rn
                from    ${ods.edw}.tbl_fact_ga_clicks_ga4
                where   lower(PagePath) like '%reference=%'
                and     WorkDate >= date '${ods.cutoff_date}'
                and     lower(concat(case when PagePath like '/%' then '' else '/' end, regexp_extract(PagePath, '^([^?#]*)', 1))) in
                        ('/retrieve-quote/', '/retrieve', '/retrieve/', '/customer-care/motor/retrieve-my-quote/', '/car-insurance/')
                ) h
        where   rn = 1
        and     Reference is not null
        and     Reference not in ('GDPR', 'NONE', 'NAN', 'NULL')
        ) x

## SourceCustomer


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_source_customer_address                                              */
/*  Description : One address (lines 1 and 2 as a salted key, Eircode area) per source customer, the hi */
/*              : ghest address id.                                                                     */
/*              :                                                                                       */
/*              : Reads: pii.addresses_assembled                                                        */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_source_customer_address as
select  SourceSystemID, SubSourceSystemID, cast(SourceCustomerID as string) as SourceCustomerID,
        ods_hash_id(concat_ws(' ', AddressLine1, AddressLine2), '')         as AddressKey,
        ods_eircode_area(Postcode)                                      as EircodeArea
from    (
        select  *, row_number() over (partition by SourceSystemID, SubSourceSystemID, SourceCustomerID
                                      order by RapierAddressId desc) as rn
        from    ${ods.pii}.addresses_assembled
        ) a
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.SourceCustomer                                                               */
/*  Description : Every customer as each source system holds it (Relay client, MFQ / QuotationStorage d */
/*              : river, travel customer ...),                                                          */
/*              : from the SCV pii tables, reduced to salted keys for matching and corroboration in ads */
/*              : : email, phone, address lines                                                         */
/*              : 1+2 (as the warehouse's address match), name + date of birth, surname + date of birth */
/*              : . Nothing personal is held:                                                           */
/*              : no names, dates of birth, addresses or contact details. RapierCustomerId (= ChillSour */
/*              : ceCustomerKey) and the SCV                                                            */
/*              : address keys join to the SCV tables. Assumes one pii row per source customer, and add */
/*              : resses joined on the same                                                             */
/*              : three columns (the pii schemas are closed to profiling: confirm on the first run).    */
/*              :                                                                                       */
/*              : Grain: one row per source customer (source system, sub source, source customer id)    */
/*              : Business key: SourceSystemId + SubSourceSystemId + SourceSystemCustomerId             */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: pii.customer_data_assembled + pii.addresses_assembled (the SCV source custome */
/*              : rs)                                                                                   */
/*              : Reads: pii.customer_data_assembled, stg.odsv2_source_customer_address                 */
/******************************************************************************************************/
create or replace table ${ods.target}.SourceCustomer as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.SourceSystemId,
                c.SubSourceSystemId,
                cast(c.SourceSystemCustomerId as string)                        as SourceSystemCustomerId,
                c.RapierCustomerId,
                c.SCV_Address_Key,
                c.ChillSourceAddressKey,
                c.CustomerGender,
                ods_hash_email(c.Email)                                             as EmailKey,
                ods_hash_phone(c.CustomerPhone)                                     as PhoneKey,
                a.AddressKey,
                coalesce(a.EircodeArea, ods_eircode_area(c.Postcode))               as EircodeArea,
                case when c.CustomerDateOfBirth is not null and nullif(trim(c.CustomerSurName), '') is not null
                     then ods_hash_id(concat_ws('|', c.CustomerFirstName, c.CustomerSurName, cast(cast(c.CustomerDateOfBirth as date) as string)), '')
                end                                                             as NameDobKey,
                case when c.CustomerDateOfBirth is not null and nullif(trim(c.CustomerSurName), '') is not null
                     then ods_hash_id(concat_ws('|', c.CustomerSurName, cast(cast(c.CustomerDateOfBirth as date) as string)), '')
                end                                                             as SurnameDobKey
        from    (
                select  *, row_number() over (partition by SourceSystemId, SubSourceSystemId, SourceSystemCustomerId
                                              order by RapierCustomerId desc) as rn
                from    ${ods.pii}.customer_data_assembled
                ) c
        left join ${ods.stg}.odsv2_source_customer_address a
                on  a.SourceSystemID = c.SourceSystemId and a.SubSourceSystemID = c.SubSourceSystemId
                and a.SourceCustomerID = cast(c.SourceSystemCustomerId as string)
        where   c.rn = 1
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "CallSession", "bk": ["ConversationId", "SessionId"], "sk": null, "nullable": [], "allow": ["AniIsMobile", "DnisIsMobile", "OriginatingDnisIsMobile"]}, {"table": "TelephonyCall", "bk": ["CallIDKey", "SeqNo"], "sk": null, "nullable": [], "allow": []}, {"table": "MessageEvent", "bk": ["SourceFeed", "CampaignId", "ProfileId", "InteractionType", "EventDateTime", "InteractionValue", "CampaignVariant", "WorkflowNode", "MessageType"], "sk": null, "nullable": ["ProfileId", "InteractionValue", "CampaignVariant", "WorkflowNode", "MessageType", "CampaignId"], "allow": []}, {"table": "Campaign", "bk": ["CampaignId", "CampaignName"], "sk": "CampaignVersionId", "nullable": [], "allow": []}, {"table": "ConsentAction", "bk": ["ActionTrackId"], "sk": null, "nullable": [], "allow": []}, {"table": "CdaMessage", "bk": ["MessageId"], "sk": null, "nullable": [], "allow": []}, {"table": "WebPageView", "bk": ["ClientID", "ClickOrder", "EffectiveDate", "EventDateTime"], "sk": null, "nullable": [], "allow": []}, {"table": "WebQuoteClick", "bk": ["GuidKey", "ClientID", "ClickOrder", "EffectiveDate"], "sk": null, "nullable": [], "allow": []}, {"table": "WebRetrieveHit", "bk": ["ClientID", "ClickOrder", "HitDateTimeUtc"], "sk": null, "nullable": [], "allow": []}, {"table": "SourceCustomer", "bk": ["SourceSystemId", "SubSourceSystemId", "SourceSystemCustomerId"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("30_Interaction", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("30_Interaction", "succeeded")
